### Ingestão de dados

In [ ]:
import sys
sys.path.append("/app")

from utils import create_spark_session, load_config

spark = create_spark_session("bronze_ordem_compras")
config = load_config()
db = config["sqlserver"]

jdbc_url = f"jdbc:sqlserver://{db['host']}:{db['port']};databaseName={db['database']}"

query = """
SELECT
    coi.ordem_compra_id,
    OC.status,
    coi.produto_id,
    P.codigo_identificacao_interno,
    P.NOME,
    P.FORNECEDOR_ID,
    FORNECEDOR.razao_social,
    coi.prazo_entrega,
    COI.quantidade,
    sld.saldo,
    COI.valor_unitario,
    COI.valor_total
FROM compras_ordens_itens coi
LEFT JOIN rel_compras_ordens_referenciadas_saldos SLD
    ON coi.ordem_compra_id = sld.ordem_compra_id
    AND coi.produto_id = sld.produto_id
    AND SLD.id_item = COI.ID
LEFT JOIN PRODUTOS P ON COI.produto_id = P.ID
LEFT JOIN PESSOAS FORNECEDOR ON FORNECEDOR.ID = P.fornecedor_id
LEFT JOIN compras_ordens OC ON COI.ordem_compra_id = OC.ID
WHERE sld.SALDO <> 0
"""

df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("query", query) \
    .option("user", db["user"]) \
    .option("password", db["password"]) \
    .option("driver", db["driver"]) \
    .option("encrypt", "true") \
    .option("trustServerCertificate", "true") \
    .load()

bronze_path = config["storage"]["bronze"]["path"]
df.write.format("delta").mode("overwrite").save(f"{bronze_path}/ordem_compras")

print(f"Bronze salvo em {bronze_path}/ordem_compras")
spark.stop()